# Benchmark Construction Pipeline

In [13]:
import json, math, re, random
import pandas as pd, numpy as np
from collections import defaultdict
from scipy.stats import norm
from dotenv import load_dotenv
from openai import OpenAI
import time
from pathlib import Path
import itertools
from tools import filter_data, exclude, count_by, sentiment_breakdown, add_shares, add_priority, apply_min_volume, rank_top, share_of, two_prop_test, chi_squared, sample_reviews

pd.set_option("display.max_colwidth", 80)
random.seed(29)

In [14]:
OUT_DIR = Path("../benchmark_outputs") 
OUT_DIR.mkdir(parents=True, exist_ok=True)

# one timestamp per run; prefixes the corpus + task files so runs don't overwrite each other
RUN_TS = time.strftime("%Y-%m-%d_%H%M%S")

FABSA = pd.read_csv("clean_fabsa.csv")
SENTS = ["positive", "negative", "neutral"]

In [15]:
# llm setup
load_dotenv()
client = OpenAI()
MODEL = "vertex_ai/gemini-2.5-flash" 

In [16]:
# cost logging
PRICE_IN, PRICE_OUT = 0.30/1e6, 2.50/1e6     # USD/token — verify current gemini-flash pricing
USAGE = defaultdict(lambda: {"calls":0, "in":0, "out":0})

def gemini(prompt, block, max_tokens=2000):
    r = client.chat.completions.create(model=MODEL,
            messages=[{"role":"user","content":prompt}], max_tokens=max_tokens)
    u = r.usage
    USAGE[block]["calls"]+=1; USAGE[block]["in"]+=u.prompt_tokens; USAGE[block]["out"]+=u.completion_tokens
    return r.choices[0].message.content

# helper - call gemini, strip json fences, parse JSON
def gemini_json(prompt, block, max_tokens=3000, retries=3):
    for attempt in range(retries):
        txt = gemini(prompt, block, max_tokens).strip()
        txt = re.sub(r"^```json|^```|```$", "", txt, flags=re.M).strip()
        try:
            return json.loads(txt)
        except json.JSONDecodeError:
            if attempt < retries - 1:
                continue
            print(f"[WARN] JSON parse failed after {retries} attempts. Raw:\n{txt[:200]}")
            raise

def cost_report():
    tot=0
    for b,d in USAGE.items():
        c=d["in"]*PRICE_IN+d["out"]*PRICE_OUT; tot+=c
        print(f"{b:12} calls={d['calls']:3d}  in={d['in']:7d}  out={d['out']:7d}  ${c:.4f}")
    print(f"{'TOTAL':12} {'':22} ${tot:.4f}")

# Step 1: plant insights by defining groups

In [17]:
# define aspect hierarchy
FABSA_INDUSTRIES = sorted(FABSA.industry.unique())

HIERARCHY = {
    "account-management": ["account-access"],
    "company-brand":      ["competitor", "general-satisfaction", "reviews"],
    "logistics-ride":     ["speed"],
    "online-experience":  ["app-website"],
    "booking-experience": ["ease-of-use"],
    "staff-support":      ["attitude-of-staff", "phone", "email"],
    "value":              ["discounts-promotions", "price-value-for-money"],
}

In [18]:
# sample 3 industries and 5 aspects

def sample_aspects(target=5):
    """Pick parents one by one, take all their children, stop when we have enough."""
    parents = list(HIERARCHY.keys())
    random.shuffle(parents)
    picked_parents, children = [], []
    for p in parents:
        picked_parents.append(p)
        children += HIERARCHY[p]
        if len(children) >= target:
            children = children[:target]
            break
    return picked_parents, children

INDUSTRIES = random.sample(FABSA_INDUSTRIES, 3)
PARENTS_USED, ASPECTS = sample_aspects(5)

print("industries:", INDUSTRIES)
print("parents:", PARENTS_USED, "-> aspects:", ASPECTS)

industries: ['Trading', 'Consulting', 'Price Comparison']
parents: ['online-experience', 'company-brand', 'staff-support'] -> aspects: ['app-website', 'competitor', 'general-satisfaction', 'reviews', 'attitude-of-staff']


In [19]:
# standardised org names per industry, e.g. Banking -> [BankA, BankB], Information Technology -> [ITA, ITB]
ORG_PREFIX = {
    "Banking": "Bank", "Consulting": "Consult", "Fashion": "Fashion", "Groceries": "Grocer",
    "Information Technology": "IT", "Price Comparison": "Price", "Ride Hailing": "Ride",
    "Streaming": "Stream", "Trading": "Trade", "Travel Booking": "Travel",
}
ORGS_PER_INDUSTRY = 2
ORGS = {ind: [f"{ORG_PREFIX[ind]}{chr(65 + i)}" for i in range(ORGS_PER_INDUSTRY)] for ind in INDUSTRIES}

for ind in INDUSTRIES:
    print(f"{ind:25s} -> {ORGS[ind]}")

Trading                   -> ['TradeA', 'TradeB']
Consulting                -> ['ConsultA', 'ConsultB']
Price Comparison          -> ['PriceA', 'PriceB']


In [25]:
# plant sentiment splits per group  (group key = industry x org x aspect; each org its own split)
MAX_REVIEWS = 1000
MIN_PER_GROUP = 35

GROUPS, gid = [], 1
raw_sizes = []
for ind in INDUSTRIES:
    for org in ORGS[ind]:
        for asp in ASPECTS:
            pos = round(random.uniform(0.20, 0.85), 2)
            neu = round(random.uniform(0.02, 0.06), 2)
            neg = round(1 - pos - neu, 4)
            raw_sizes.append((ind, org, asp, {"positive": pos, "negative": neg, "neutral": neu},
                              random.randint(45, 65)))

# guarantee minimum, distribute remaining budget proportionally
floor_total = MIN_PER_GROUP * len(raw_sizes)
remaining = MAX_REVIEWS - floor_total
raw_total = sum(r[4] for r in raw_sizes)
for ind, org, asp, shares, raw_n in raw_sizes:
    bonus = int(round(raw_n / raw_total * remaining)) if remaining > 0 else 0
    GROUPS.append((f"G{gid:04d}", ind, org, asp, shares, MIN_PER_GROUP + bonus))
    gid += 1
GRP = {g[0]: g for g in GROUPS}

print(f"groups: {len(GROUPS)} | budget: {MAX_REVIEWS} | actual: {sum(g[5] for g in GROUPS)} | min group: {min(g[5] for g in GROUPS)}")
for g in GROUPS[:3]:
    print(f"{g[0]}  {g[1]:22s} {g[2]:9s} {g[3]:18s}  pos={g[4]['positive']:.0%}  neg={g[4]['negative']:.0%}  neu={g[4]['neutral']:.0%}  n={g[5]}")

groups: 30 | budget: 1000 | actual: 1050 | min group: 35
G0001  Trading                TradeA    app-website         pos=33%  neg=62%  neu=5%  n=35
G0002  Trading                TradeA    competitor          pos=36%  neg=61%  neu=3%  n=35
G0003  Trading                TradeA    general-satisfaction  pos=60%  neg=35%  neu=5%  n=35


In [26]:
# expand groups into cells (exact counts)
def expand(g):
    _, ind, org, asp, sh, n = g
    c = {s: int(round(sh.get(s, 0) * n)) for s in SENTS}
    r = n - sum(c.values())
    if r: c[max(c, key=c.get)] += r      # push rounding residue onto largest bucket
    return [(ind, org, asp, s, c[s], g[0]) for s in SENTS if c[s] > 0]

CELLS = [c for g in GROUPS for c in expand(g)]
print(len(GROUPS), "groups ->", len(CELLS), "cells ->", sum(c[4] for c in CELLS), "reviews to generate")

30 groups -> 90 cells -> 1050 reviews to generate


In [27]:
# save planted insights: 1 row per org group + 1 row per industry group (deduped).
#   org group      = (industry, org, aspect) -> that org's planted split
#   industry group = (industry, aspect)      -> volume-weighted aggregate of its orgs
ind_totals = defaultdict(lambda: {"positive": 0.0, "negative": 0.0, "neutral": 0.0, "n": 0})
for _gid, ind, org, asp, sh, n in GROUPS:
    agg = ind_totals[(ind, asp)]
    for s in SENTS:
        agg[s] += sh[s] * n
    agg["n"] += n

org_rows = [{"group_id": _gid, "level": "org", "industry": ind, "org": org, "child_aspect": asp,
             "pos_share": round(sh["positive"], 3), "neg_share": round(sh["negative"], 3),
             "neu_share": round(sh["neutral"], 3), "n": n}
            for _gid, ind, org, asp, sh, n in GROUPS]

industry_rows = [{"group_id": "", "level": "industry", "industry": ind, "org": "ALL", "child_aspect": asp,
                  "pos_share": round(a["positive"] / a["n"], 3), "neg_share": round(a["negative"] / a["n"], 3),
                  "neu_share": round(a["neutral"] / a["n"], 3), "n": a["n"]}
                 for (ind, asp), a in ind_totals.items()]

INSIGHTS = pd.DataFrame(org_rows + industry_rows)
insights_path = OUT_DIR / f"{RUN_TS}_insights.csv"
INSIGHTS.to_csv(insights_path, index=False)
print(f"wrote {insights_path.name}: {len(org_rows)} org rows + {len(industry_rows)} industry rows = {len(INSIGHTS)}")
INSIGHTS.head()

wrote 2026-07-17_132626_insights.csv: 30 org rows + 15 industry rows = 45


,group_id,level,industry,org,child_aspect,pos_share,neg_share,neu_share,n
0,G0001,org,Trading,TradeA,app-website,0.33,0.62,0.05,35
1,G0002,org,Trading,TradeA,competitor,0.36,0.61,0.03,35
2,G0003,org,Trading,TradeA,general-satisfaction,0.60,0.35,0.05,35
3,G0004,org,Trading,TradeA,reviews,0.36,0.61,0.03,35
4,G0005,org,Trading,TradeA,attitude-of-staff,0.71,0.27,0.02,35


# Step 2: generate reviews according to the pre-defined patterns

In [28]:
def fabsa_seeds(ind, asp, sent, k=3):
    m = FABSA[(FABSA.industry==ind) & (FABSA.child_aspect==asp) & (FABSA.sentiment==sent)]
    return m.text.dropna().drop_duplicates().head(k).tolist()

# generate reviews for one (industry, org, aspect, sentiment) cell.
# returns (named, plain): `named` mention the org by name, `plain` mention no company at all.
def gen_reviews(ind, org, asp, sent, n_named, n_plain, seeds):
    ex = "\n".join(f"- {s}" for s in seeds) or "- (none)"
    out = gemini_json(
      f"Write short customer reviews for '{org}', a {ind} company, each expressing "
      f"{sent.upper()} sentiment about '{asp}'. Match the style of these real examples:\n{ex}\n"
      f"Rules: each review MUST be 5-30 words and vary the wording. Return ONLY a JSON object with:\n"
      f"  'named': a list of {n_named} reviews that naturally mention the company by name '{org}'.\n"
      f"  'plain': a list of {n_plain} reviews that do NOT mention any company or brand name.",
      block="generate")
    if not isinstance(out, dict):
        out = {"named": [], "plain": out if isinstance(out, list) else []}
    named = [r for r in out.get("named", []) if 5 <= len(r.split()) <= 30 and org in r]
    plain = [r for r in out.get("plain", []) if 5 <= len(r.split()) <= 30 and org not in r]
    return named, plain

# rewrite reviews to add realistic contextual noise (a brief off-topic aside / casual filler /
# passing mention of something unrelated) WITHOUT changing the labelled opinion or adding/removing
# company names. Used on ~10% of the corpus to make the language less templated.
def add_noise(reviews, asp, sent):
    if not reviews:
        return []
    out = gemini_json(
      f"Rewrite each customer review to sound more natural and realistic by weaving in a little "
      f"contextual noise — a brief off-topic aside, casual filler, or a passing mention of something "
      f"unrelated — while KEEPING the same clear {sent.upper()} opinion about '{asp}' as the main "
      f"point. Do NOT add or remove any company or brand names. Each rewrite must be 8-45 words. "
      f"Return ONLY a JSON list of strings, in the same order.\n{json.dumps(reviews)}", block="noise")
    return out if isinstance(out, list) else reviews

# independent Gemini call to verify aspect and sentiment labels
def verify(reviews, asp, sent):
    if not reviews:
        return []
    return gemini_json(
      f"For each review answer true only if it expresses {sent.upper()} sentiment about "
      f"'{asp}', else false. Return ONLY a JSON list of booleans, same order.\n"
      f"{json.dumps(reviews)}", block="verify")

In [29]:
# build the corpus (slow) -- per cell: ~20% of reviews name the org, the rest none.
# NOTE: contextual-noise injection is commented out for now (see block below).
start = time.time()
NAME_FRAC = 0.20
rows, rid = [], 0
for i, (ind, org, asp, sent, n, gid) in enumerate(CELLS, 1):
    n_named = round(NAME_FRAC * n); n_plain = n - n_named
    seeds = fabsa_seeds(ind, asp, sent)
    named, plain = [], []
    for _ in range(2):
        need_named = n_named - len(named); need_plain = n_plain - len(plain)
        if need_named <= 0 and need_plain <= 0: break
        gn, gp = gen_reviews(ind, org, asp, sent, need_named + 1, need_plain + 1, seeds)
        batch = [(r, True) for r in gn] + [(r, False) for r in gp]
        ok = verify([r for r, _ in batch], asp, sent)
        for (r, _), good in zip(batch, ok):
            if not good: continue
            if org in r and len(named) < n_named: named.append(r)
            elif org not in r and len(plain) < n_plain: plain.append(r)
    kept = named[:n_named] + plain[:n_plain]

    # --- contextual noise injection (DISABLED for now) ---
    # to re-enable: first (re-)run the review-generation cell so `add_noise` is defined, then uncomment.
    # NOISE_FRAC = 0.10
    # n_noisy = round(NOISE_FRAC * len(kept))
    # if n_noisy:
    #     step = len(kept) / n_noisy
    #     idx = sorted({int(j * step) for j in range(n_noisy)})
    #     noised = add_noise([kept[j] for j in idx], asp, sent)
    #     okn = verify(noised, asp, sent)
    #     for j, new, good in zip(idx, noised, okn):
    #         new = str(new)
    #         if good and 5 <= len(new.split()) <= 45 and (org in kept[j]) == (org in new):
    #             kept[j] = new
    # -----------------------------------------------------

    for t in kept:
        rows.append((rid, ind, org, asp, sent, t, gid)); rid += 1
    print(f"[{i}/{len(CELLS)}] {gid} {org} {sent}: {len(kept)}/{n}")

CORPUS = pd.DataFrame(rows, columns=["review_id","industry","org","child_aspect","sentiment","text","group_id"])
corpus_path = OUT_DIR / f"{RUN_TS}_corpus.csv"
CORPUS.to_csv(corpus_path, index=False)

elapsed = time.time() - start
mention = CORPUS.apply(lambda x: x["org"] in x["text"], axis=1).mean() if len(CORPUS) else 0
print(f"\ncorpus: {len(CORPUS)} reviews ({mention:.0%} name their org) in {elapsed/60:.1f} min -> {corpus_path.name}")
cost_report()

[1/90] G0001 TradeA positive: 12/12
[2/90] G0001 TradeA negative: 21/21
[3/90] G0001 TradeA neutral: 2/2
[4/90] G0002 TradeA positive: 3/13
[5/90] G0002 TradeA negative: 6/21
[6/90] G0002 TradeA neutral: 1/1
[7/90] G0003 TradeA positive: 21/21
[8/90] G0003 TradeA negative: 12/12
[9/90] G0003 TradeA neutral: 2/2
[10/90] G0004 TradeA positive: 13/13
[11/90] G0004 TradeA negative: 21/21
[12/90] G0004 TradeA neutral: 1/1
[13/90] G0005 TradeA positive: 25/25
[14/90] G0005 TradeA negative: 9/9
[15/90] G0005 TradeA neutral: 1/1
[16/90] G0006 TradeB positive: 16/16
[17/90] G0006 TradeB negative: 18/18
[18/90] G0006 TradeB neutral: 1/1
[19/90] G0007 TradeB positive: 14/14
[20/90] G0007 TradeB negative: 4/19
[21/90] G0007 TradeB neutral: 2/2
[22/90] G0008 TradeB positive: 8/8
[23/90] G0008 TradeB negative: 26/26
[24/90] G0008 TradeB neutral: 1/1
[25/90] G0009 TradeB positive: 22/22
[26/90] G0009 TradeB negative: 12/12
[27/90] G0009 TradeB neutral: 1/1
[28/90] G0010 TradeB positive: 11/11


BadRequestError: Error code: 400 - {'error': {'message': 'Budget has been exceeded! Current cost: 500.00313303999604, Max budget: 500', 'type': 'budget_exceeded', 'param': None, 'code': '400'}}

In [ ]:
CORPUS.head()

,review_id,industry,child_aspect,sentiment,text,group_id
0,0,Consulting,ease-of-use,positive,ORG made the entire process incredibly simple. Highly recommend!,G0001
1,1,Consulting,ease-of-use,positive,Getting started with ORG was surprisingly easy. Great service!,G0001
2,2,Consulting,ease-of-use,positive,Their platform is so user-friendly. A joy to work with.,G0001
3,3,Consulting,ease-of-use,positive,No complications at all. ORG's system is very intuitive.,G0001
4,4,Consulting,ease-of-use,positive,Everything was explained clearly and easy to understand.,G0001


# Step 3a: create task specs 

In [ ]:
# build solver functions for each task.
# each solver runs the tools on the corpus to produce the gold answer + records the tool calls.
# `where` selects a segment: {"industry": name} or {"org": name}. filter is implicit (not a step).

# ---- 1-step solvers (descriptive) ----

def solve_count(where, asp, sent):
    # share_of returns {n, count, share}; read the count for a "how many" answer
    r = share_of(filter_data(CORPUS, **where, child_aspect=asp), sent)
    return r["count"], [
        {"tool":"share_of","args":{**where,"child_aspect":asp,"sentiment":sent}}]

def solve_share(where, asp, sent):
    r = share_of(filter_data(CORPUS, **where, child_aspect=asp), sent)
    return r["share"], [
        {"tool":"share_of","args":{**where,"child_aspect":asp,"sentiment":sent}}]

# ---- 2-step solvers ----

def solve_top_k(where, k, by):
    filt = {**where, **({"sentiment":"negative"} if by=="negative" else {})}
    top = rank_top(count_by(filter_data(CORPUS, **filt), "child_aspect"), by="count", top_n=k)
    return top.child_aspect.tolist(), [
        {"tool":"count_by","args":{**filt,"group_by":"child_aspect"}},
        {"tool":"rank_top","args":{"by":"count","top_n":k}}]

def solve_split(where, group_by):
    sh = add_shares(sentiment_breakdown(filter_data(CORPUS, **where), group_by))
    v = {k: float(sh[k].iloc[0]) for k in ["pos_share","neg_share","neu_share"]}
    return v, [
        {"tool":"sentiment_breakdown","args":{**where,"group_by":group_by}},
        {"tool":"add_shares","args":{}}]

def solve_compare(where_a, where_b, asp, sent):
    ra = share_of(filter_data(CORPUS, **where_a, child_aspect=asp), sent)
    rb = share_of(filter_data(CORPUS, **where_b, child_aspect=asp), sent)
    return {"a":ra["share"],"b":rb["share"],"higher":"a" if (ra["share"] or 0)>(rb["share"] or 0) else "b"}, [
        {"tool":"share_of","args":{**where_a,"child_aspect":asp,"sentiment":sent}},
        {"tool":"share_of","args":{**where_b,"child_aspect":asp,"sentiment":sent}}]

# ---- 3-step solver ----

def solve_two_prop(where_a, where_b, asp, sent):
    da = filter_data(CORPUS, **where_a, child_aspect=asp)
    db = filter_data(CORPUS, **where_b, child_aspect=asp)
    r = two_prop_test(da, db, sent)
    return r, [
        {"tool":"share_of","args":{**where_a,"child_aspect":asp,"sentiment":sent}},
        {"tool":"share_of","args":{**where_b,"child_aspect":asp,"sentiment":sent}},
        {"tool":"two_prop_test","args":{"sentiment":sent}}]

# ---- 4-step solvers (diagnostic) ----

def solve_driver(where, k):
    # L6: rank child aspects by negative share (severity), with a volume guard
    sh = add_shares(sentiment_breakdown(filter_data(CORPUS, **where), "child_aspect"))
    vol = apply_min_volume(sh, min_volume=5)
    top = rank_top(vol, by="neg_share", top_n=k)
    v = list(zip(top.child_aspect, top.neg_share.round(4)))
    return v, [
        {"tool":"sentiment_breakdown","args":{**where,"group_by":"child_aspect"}},
        {"tool":"add_shares","args":{}},
        {"tool":"apply_min_volume","args":{"min_volume":5}},
        {"tool":"rank_top","args":{"by":"neg_share","top_n":k}}]

def solve_prioritise(where, k):
    # L7: rank child aspects by priority = complaint volume x severity (prioritisation quadrant)
    sh = add_priority(add_shares(sentiment_breakdown(filter_data(CORPUS, **where), "child_aspect")))
    top = rank_top(sh, by="priority", top_n=k)
    v = list(zip(top.child_aspect, top.priority.round(4)))
    return v, [
        {"tool":"sentiment_breakdown","args":{**where,"group_by":"child_aspect"}},
        {"tool":"add_shares","args":{}},
        {"tool":"add_priority","args":{}},
        {"tool":"rank_top","args":{"by":"priority","top_n":k}}]

SOLVERS = {
    "solve_count": solve_count, "solve_share": solve_share,
    "solve_top_k": solve_top_k, "solve_split": solve_split,
    "solve_compare": solve_compare, "solve_two_prop": solve_two_prop,
    "solve_driver": solve_driver, "solve_prioritise": solve_prioritise,
} 

Task structure:
- tid: task ID, numbered by increasing difficulty
- type: question intent (descriptive, diagnostic or prescriptive)
- inds: industries
- solver: function that computes the gold answer 
- args: arguments to pass to that solver
- spec: structured description of the task, input to Gemini to write the question
- steps: number of tool calls in the gold path

In [ ]:
# generate task specs — 3 tasks per level, 10 levels => 30 tasks total.
#   Segment = `where` dict {"industry": name} or {"org": name}; filter is implicit.
#   industry-level: L1-L5 (per_level each).   org-level: L6-L10 (one per focus org = ORGS[ind][0]).
#   Per the taxonomy, L6-L10 are organisation-specific (one org); the industry is used only as an
#   internal "industry average" reference inside L8/L10. Prescriptive tasks (L8/L9/L10) compose
#   `components` = a list of {key, solver, args, spec, steps}; the builder runs each to get its
#   finding + tool path.
TASKS_PER_LEVEL = 3

def auto_tasks(industries, aspects, seed=42, per_level=TASKS_PER_LEVEL):
    rng = random.Random(seed)
    T = []

    def _add(tid, typ, inds, solver, args, spec, steps):
        T.append((tid, typ, set(inds), solver, args, spec, steps))

    def _comp(key, solver, args, op, steps):
        return {"key": key, "solver": solver, "args": args, "spec": {"op": op}, "steps": steps}

    def _compose(tid, ind, fo, op, components, spec_extra=None):
        steps = sum(c["steps"] for c in components) + 1   # + sample_reviews evidence
        spec = {"op": op, "scope": "org", "subject": fo, "where": {"org": fo}, **(spec_extra or {})}
        _add(tid, "prescriptive", [ind], "solve_compose", (components,), spec, steps)

    pairs = list(itertools.combinations(industries, 2))
    sents = ["positive", "negative"]

    # ============ INDUSTRY-LEVEL: L1-L5 (3 each) ============
    di = gi = 0

    # L1 count / share
    pool = [(ind, asp) for ind in industries for asp in aspects]
    rng.shuffle(pool)
    l1_ops = [("count", "negative"), ("share", "positive"), ("share", "negative")]
    for j in range(per_level):
        ind, asp = pool[j % len(pool)]
        op, sent = l1_ops[j % len(l1_ops)]
        di += 1
        _add(f"DESC-{di:02d}", "descriptive", [ind], f"solve_{op}", ({"industry": ind}, asp, sent),
             {"op": op, "scope": "industry", "subject": ind, "aspect": asp, "sent": sent}, 1)

    # L2 split
    for j in range(per_level):
        ind = industries[j % len(industries)]
        di += 1
        _add(f"DESC-{di:02d}", "descriptive", [ind], "solve_split", ({"industry": ind}, "industry"),
             {"op": "split", "scope": "industry", "subject": ind}, 2)

    # L3 top_k
    l3 = [("volume", 1), ("negative", 3), ("negative", 2)]
    for j in range(per_level):
        ind = industries[j % len(industries)]
        by, k = l3[j % len(l3)]
        di += 1
        _add(f"DESC-{di:02d}", "descriptive", [ind], "solve_top_k", ({"industry": ind}, k, by),
             {"op": f"top_{by}", "scope": "industry", "subject": ind, **({"k": k} if k > 1 else {})}, 2)

    # L4 compare
    used = set()
    for j in range(per_level):
        pa, pb = pairs[j % len(pairs)]
        asp = rng.choice([a for a in aspects if a not in used] or aspects); used.add(asp)
        sent = sents[j % len(sents)]
        gi += 1
        _add(f"DIAG-{gi:02d}", "diagnostic", [pa, pb], "solve_compare",
             ({"industry": pa}, {"industry": pb}, asp, sent),
             {"op": "compare", "scope": "industry", "a": pa, "b": pb, "aspect": asp, "sent": sent}, 2)

    # L5 two_prop
    used = set()
    for j in range(per_level):
        pa, pb = pairs[j % len(pairs)]
        asp = rng.choice([a for a in aspects if a not in used] or aspects); used.add(asp)
        sent = rng.choice(sents)
        gi += 1
        _add(f"DIAG-{gi:02d}", "diagnostic", [pa, pb], "solve_two_prop",
             ({"industry": pa}, {"industry": pb}, asp, sent),
             {"op": "test", "scope": "industry", "a": pa, "b": pb, "aspect": asp, "sent": sent}, 3)

    # ============ ORG-LEVEL: L6-L10 (organisation-specific, one focus org per industry) ============
    ogi = opi = 0
    for ind in industries:
        fo = ORGS[ind][0]                                  # focus org (organisation-specific)
        wfo, wind = {"org": fo}, {"industry": ind}

        # L6 driver identification
        ogi += 1
        _add(f"ODIAG-{ogi:02d}", "diagnostic", [ind], "solve_driver", (wfo, 2),
             {"op": "driver", "scope": "org", "subject": fo, "k": 2}, 4)
        # L7 prioritisation (volume x severity)
        ogi += 1
        _add(f"ODIAG-{ogi:02d}", "diagnostic", [ind], "solve_prioritise", (wfo, 3),
             {"op": "prioritise", "scope": "org", "subject": fo, "k": 3}, 4)

        # L8 comparative diagnosis: org drivers vs the industry-average drivers
        opi += 1
        _compose(f"OPRES-{opi:02d}", ind, fo, "compare_diag", [
            _comp("org_drivers", "solve_driver", (wfo, 3), "driver", 4),
            _comp("industry_avg_drivers", "solve_driver", (wind, 3), "driver", 4),
        ], {"industry": ind})
        # L9 targeted recommendation: overview + gap vs the industry average
        asp9, sent9 = rng.choice(aspects), rng.choice(sents)
        opi += 1
        _compose(f"OPRES-{opi:02d}", ind, fo, "recommend", [
            _comp("overview", "solve_split", (wfo, "org"), "split", 2),
            _comp("vs_industry_avg", "solve_two_prop", (wfo, wind, asp9, sent9), "test", 3),
        ])
        # L10 full CX report: multiple descriptive + multiple diagnostic
        asp10, sent10 = rng.choice(aspects), rng.choice(sents)
        opi += 1
        _compose(f"OPRES-{opi:02d}", ind, fo, "report", [
            _comp("overview", "solve_split", (wfo, "org"), "split", 2),
            _comp("top_complaints", "solve_top_k", (wfo, 3, "negative"), "top_negative", 2),
            _comp("drivers", "solve_driver", (wfo, 3), "driver", 4),
            _comp("priorities", "solve_prioritise", (wfo, 3), "prioritise", 4),
            _comp("vs_industry_avg", "solve_two_prop", (wfo, wind, asp10, sent10), "test", 3),
        ])

    return T

T = auto_tasks(INDUSTRIES, ASPECTS, seed=42)
n_ind = sum(t[5].get("scope") == "industry" for t in T)
n_org = sum(t[5].get("scope") == "org" for t in T)
print(f"{len(T)} tasks ({TASKS_PER_LEVEL}/level x 10 levels): {n_ind} industry (L1-L5) + {n_org} org (L6-L10)\n")
for tid, typ, inds, solver, args, spec, steps in T:
    print(f"{tid:10s} {steps:2d}-step  {spec['scope']:8s} {typ:12s}  {spec['op']:12s} subj={spec.get('subject') or spec.get('a')}")

# Step 3b: derive gold answers (JSON) and gold tool paths

In [ ]:
# run solvers to get gold answers and paths (descriptive and diagnostic)
records = {}
for tid, type, inds, solver_name, args, spec, steps in T:
    if solver_name == "solve_compose":
        continue
    fn = SOLVERS[solver_name]
    ans, path = fn(*args)
    assert len(path) == steps, f"{tid}: expected {steps} steps, got {len(path)}"
    records[tid] = dict(task_id=tid, type=type, steps=steps,
                        industries=sorted(inds), spec=spec,
                        gold_answer=ans, gold_tool_path=path)

for tid in sorted(records)[:5]:
    print(f"{tid} ({records[tid]['steps']}-step): {records[tid]['gold_answer']}")

DESC-01 (1-step): 14
DESC-02 (1-step): 0.7
DESC-03 (1-step): 0.725
DESC-04 (1-step): 0.333
DESC-05 (2-step): ['app-website', 'account-access', 'ease-of-use']


In [ ]:
# build prescriptive tasks (L8 comparative diagnosis, L9 recommendation, L10 full report).
# run each component analysis, concatenate their tool paths, then sample_reviews for evidence.
# gold_answer = {"findings": {component_key: answer}}; sampled reviews are evidence, not graded.
for tid, type, inds, solver_name, args, spec, steps in T:
    if solver_name != "solve_compose":
        continue
    components = args[0]
    path, findings, meta = [], {}, []
    for c in components:
        ans, cpath = SOLVERS[c["solver"]](*c["args"])
        assert len(cpath) == c["steps"], f"{tid}/{c['key']}: {len(cpath)} != {c['steps']}"
        findings[c["key"]] = ans
        path += cpath
        meta.append({"key": c["key"], "spec": c["spec"], "steps": c["steps"]})
    path += [{"tool": "sample_reviews", "args": {**spec["where"], "sentiment": "negative", "n": 3}}]
    assert len(path) == steps, f"{tid}: {len(path)} != {steps}"
    records[tid] = dict(task_id=tid, type=type, steps=steps, industries=sorted(inds),
                        spec={**spec, "components": meta},
                        gold_answer={"findings": findings}, gold_tool_path=path)
    print(f"{tid} ({steps}-step, {spec['op']}): {[c['key'] for c in components]} + sample_reviews")

# Step 3c: verify gold tool paths and answers

Sanity check: independently **re-run each recorded `gold_tool_path`** against `CORPUS`
(calling the real tool functions directly, *not* the solver functions that produced them) and
confirm it reproduces `gold_answer`. A wrong tool name or argument in a path would make the two
diverge. The final answer is read off the last tool's output, shaped per the task's `spec["op"]`.

In [ ]:
# self-contained: load the latest saved corpus + tasks CSV, then re-run every recorded
# gold_tool_path against CORPUS with the real tools and check it reproduces the gold answer.
# gold_answer_json is the deterministic column; gold_answer_text is the NL answer (not checked).
#   count -> share_of[...]["count"]   share -> share_of[...]["share"]
#   prescriptive (recommend/compare_diag/report): findings read by slicing the path per component.
import ast, json

def _parse(s):  # gold_answer_json is JSON; spec/tool_path are Python-repr -> try JSON then literal_eval
    s = str(s)
    try:
        return json.loads(s)
    except Exception:
        return ast.literal_eval(s)

# corpus + task files are timestamp-prefixed per run; load the most recent of each.
_corpus_path = max(OUT_DIR.glob("*corpus.csv"), key=lambda p: p.stat().st_mtime)
_tasks_path = max(OUT_DIR.glob("*tasks.csv"), key=lambda p: p.stat().st_mtime)
print(f"loading {_corpus_path.name} + {_tasks_path.name}")
CORPUS = pd.read_csv(_corpus_path)

_tasks = pd.read_csv(_tasks_path)
records = {}
for _, r in _tasks.iterrows():
    records[r["task_id"]] = {
        "task_id": r["task_id"],
        "steps": int(r["steps"]),
        "spec": ast.literal_eval(r["spec"]),
        "gold_answer": _parse(r.get("gold_answer_json", r.get("gold_answer"))),
        "gold_tool_path": ast.literal_eval(r["gold_tool_path"]),
        "question": r["question"],
    }

def _slice(args, keys):
    return filter_data(CORPUS, **{k: args[k] for k in keys if k in args})

# key sets exclude 'sentiment' where a tool must NOT pre-filter by sentiment (share_of, breakdown)
_SEG = ("industry", "org", "child_aspect")               # segment slice (no sentiment)
_SEG_S = ("industry", "org", "child_aspect", "sentiment")  # segment slice incl. sentiment
_COMPOSE_OPS = ("recommend", "compare_diag", "report")

def run_gold_path(path, spec, records):
    op = spec["op"]

    # prescriptive: findings = each component's answer, read by slicing the path per component
    if op in _COMPOSE_OPS:
        findings, idx = {}, 0
        for comp in spec["components"]:
            n = comp["steps"]
            findings[comp["key"]] = run_gold_path(path[idx:idx + n], comp["spec"], records)
            idx += n
        return {"findings": findings}

    cur = None                            # current DataFrame for chained ops
    share_slices, share_results = [], []  # from share_of steps -> feed count/share/compare/two_prop
    tp = None
    for step in path:
        t, a = step["tool"], step["args"]
        if t == "share_of":
            df = _slice(a, _SEG)
            share_slices.append(df)
            share_results.append(share_of(df, a["sentiment"]))
        elif t == "count_by":
            cur = count_by(_slice(a, _SEG_S + ("parent_aspect",)), a["group_by"])
        elif t == "sentiment_breakdown":
            cur = sentiment_breakdown(_slice(a, _SEG), a["group_by"])
        elif t == "add_shares":
            cur = add_shares(cur)
        elif t == "add_priority":
            cur = add_priority(cur)
        elif t == "apply_min_volume":
            cur = apply_min_volume(cur, min_volume=a.get("min_volume", 30))
        elif t == "rank_top":
            cur = rank_top(cur, by=a["by"], top_n=a["top_n"])
        elif t == "two_prop_test":
            tp = two_prop_test(share_slices[-2], share_slices[-1], a["sentiment"])
        elif t == "sample_reviews":
            sample_reviews(_slice(a, _SEG_S), n=a.get("n", 3))  # evidence only
        else:
            raise ValueError(f"unknown tool in path: {t}")

    # read the final answer off the last tool's output, shaped per task op
    if op == "count":
        return share_results[-1]["count"]
    if op == "share":
        return share_results[-1]["share"]
    if op in ("top_volume", "top_negative"):
        return cur["child_aspect"].tolist()
    if op == "split":
        return {k: float(cur[k].iloc[0]) for k in ("pos_share", "neg_share", "neu_share")}
    if op == "compare":
        a_, b_ = share_results[0]["share"], share_results[1]["share"]
        return {"a": a_, "b": b_, "higher": "a" if (a_ or 0) > (b_ or 0) else "b"}
    if op == "test":
        return tp
    if op == "driver":
        return [[asp, float(round(ns, 4))] for asp, ns in zip(cur["child_aspect"], cur["neg_share"])]
    if op == "prioritise":
        return [[asp, float(round(p, 4))] for asp, p in zip(cur["child_aspect"], cur["priority"])]
    raise ValueError(f"unknown op: {op}")


def _match(a, b, tol=1e-4):
    """Tolerant structural comparison: list==tuple, floats within tol, exact otherwise."""
    if isinstance(a, bool) or isinstance(b, bool):
        return a == b
    if a is None or b is None:
        return a is None and b is None
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return math.isclose(float(a), float(b), abs_tol=tol)
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_match(a[k], b[k], tol) for k in a)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_match(x, y, tol) for x, y in zip(a, b))
    return a == b


print(f"Verifying {len(records)} gold tool paths against CORPUS ({len(CORPUS)} reviews)")
print("=" * 78)
n_pass = 0
for tid, rec in records.items():
    got = run_gold_path(rec["gold_tool_path"], rec["spec"], records)
    ok = _match(got, rec["gold_answer"])
    n_pass += ok
    print(f"\n[{tid}] {rec['steps']}-step  {rec['question']}")
    print(f"  gold answer : {rec['gold_answer']}")
    print(f"  path re-ran : {got}")
    print(f"  {'PASS' if ok else 'FAIL <<<<<<'}")

print("\n" + "=" * 78)
print(f"{n_pass}/{len(records)} tasks: gold_tool_path reproduces gold_answer")

# Step 4: generate task questions

In [ ]:
# use LLM to write questions from task specs
start = time.time()
def brief(sp):
    o = sp["op"]; who = sp.get("subject")
    if o == "count":        return f"How many complaints {who} received about {sp['aspect']}."
    if o == "share":        return f"The share of {who} feedback on {sp['aspect']} that is {'praise' if sp['sent']=='positive' else 'complaints'}."
    if o == "top_volume":   return f"Which single topic {who} customers mention most."
    if o == "top_negative": return f"The {sp['k']} topics with the most complaints for {who}."
    if o == "split":        return f"The overall breakdown of happy vs unhappy {who} customers."
    if o == "compare":      return f"Whether {sp['a']} or {sp['b']} customers have more {'complaints about' if sp['sent']=='negative' else 'praise for'} {sp['aspect']}."
    if o == "test":         return f"Whether the difference in {'complaints about' if sp['sent']=='negative' else 'praise for'} {sp['aspect']} between {sp['a']} and {sp['b']} is statistically significant."
    if o == "driver":       return (f"The single biggest driver of dissatisfaction for {who}." if sp['k']==1
                                    else f"The top {sp['k']} drivers of dissatisfaction for {who}.")
    if o == "prioritise":   return f"The top {sp['k']} issues {who} should fix first, weighing both how many customers are affected and how unhappy they are."
    if o == "compare_diag": return f"How {who}'s biggest complaint drivers compare with the {sp['industry']} industry average, and where {who} does worse."
    if o == "recommend":    return f"Diagnose the biggest customer problems for {who} and recommend the top fixes."
    if o == "report":       return f"A full customer-experience report for {who}: overall happiness, top complaints, biggest drivers, what to prioritise, and how it compares to the industry average."

briefs = {tid: brief(r["spec"]) for tid, r in records.items()}

qs = gemini_json(
  "You are a CX analytics lead writing questions for a business intelligence tool.\n\n"
  "For each item below, write ONE clear, specific, realistic question that a business "
  "stakeholder would  ask. The question must be answerable using exactly the "
  "analysis described in the item.\n\n"

  "LANGUAGE:\n"
  "- Use plain, professional business English.\n"
  "- Talk about 'complaints', 'issues', 'what customers like', 'praise', 'frustrations' etc.\n"

  "TRANSLATE TOPICS INTO NATURAL PHRASING:\n"
  "- 'app-website'          -> 'the app or website'\n"
  "- 'ease-of-use'          -> 'how easy the service is to use'\n"
  "- 'account-access'       -> 'accessing their account'\n"
  "- 'price-value-for-money'-> 'pricing and value for money'\n"
  "(Apply the same natural-language treatment to any other topic.)\n\n"

  "NAMES AND COMPARISONS:\n"
  "- Some items refer to a whole industry (e.g. 'Banking'); others to a specific company "
  "within an industry (e.g. 'BankA', 'ITA'). Always keep the exact name given.\n"
  "- When comparing one company against the rest of its industry, use the phrase "
  "'the industry average'.\n"
  "- Preserve any specific counts, such as 'top 3'.\n\n"

  "OUTPUT:\n"
  "Return ONLY a JSON object mapping each item's id to its question string.\n\n"
  + json.dumps(briefs), block="questions", max_tokens=2000)

for tid, q in qs.items():
    records[tid]["question"] = q
    print(f"{tid}: {q}")

elapsed = time.time() - start
print(f"\nquestions generated in {elapsed:.1f}s")
cost_report()

# Step 5: generate gold answers (text)

In [ ]:
# business-facing natural-language gold answer (generated AFTER the JSON answer + question).
# Each NL answer must use ONLY the figures/insights in the JSON gold answer and answer that task's
# question directly. Stored as records[tid]["gold_answer_nl"] -> a second gold-answer column.
start = time.time()
nl_inputs = {tid: {"question": r["question"], "gold_answer": r["gold_answer"]}
             for tid, r in records.items()}

nl = gemini_json(
  "You are a CX analytics lead writing the answer a stakeholder receives.\n\n"
  "For each item you are given the stakeholder's QUESTION and the correct ANSWER as "
  "structured JSON (the ground truth). Write ONE clear, business-facing answer that "
  "directly answers the question.\n\n"

  "MOST IMPORTANT RULE:\n"
  "Use ONLY the figures and findings in the JSON answer. Never invent numbers, add "
  "detail, or state anything the JSON does not support. If it isn't in the JSON, it "
  "does not go in the answer.\n\n"

  "LANGUAGE:\n"
  "- Plain, professional business English.\n"

  "NUMBERS:\n"
  "- Render proportions as whole-number percentages (0.725 -> '73%').\n"
  "- Keep counts as whole numbers.\n\n"

  "TRANSLATE TOPICS INTO NATURAL PHRASING:\n"
  "- 'app-website'           -> 'the app or website'\n"
  "- 'ease-of-use'           -> 'how easy the service is to use'\n"
  "- 'account-access'        -> 'signing in or accessing their account'\n"
  "- 'attitude-of-staff'     -> 'staff attitude'\n"
  "- 'price-value-for-money' -> 'value for money'\n"
  "- 'discounts-promotions'  -> 'discounts and promotions'\n\n"

  "NAMES AND COMPARISONS:\n"
  "- Use the exact company and industry names from the JSON.\n"
  "- When the JSON compares a company to the rest of its industry, use the phrase "
  "'the industry average'.\n\n"

  "ANSWER SHAPE BY TASK TYPE:\n"
  "- Significance test: state whether the difference is statistically significant "
  "(p < 0.05) and which side is higher.\n"
  "- Report or recommendation: synthesise the findings into a short paragraph that "
  "ends with the single most important action to take.\n"
  "- Everything else: 1-2 sentences that answer the question directly.\n\n"

  "OUTPUT:\n"
  "Return ONLY a JSON object mapping each item's id to its answer string.\n\n"
  + json.dumps(nl_inputs, default=str), block="answers", max_tokens=6000)

for tid, a in nl.items():
    records[tid]["gold_answer_nl"] = a
    print(f"{tid}: {a}")

elapsed = time.time() - start
print(f"\nnatural-language answers generated in {elapsed:.1f}s")
cost_report()

In [ ]:
# save tasks.csv and print cost report.  Pipeline outputs (all timestamp-prefixed):
#   {RUN_TS}_corpus.csv (Step 2) | {RUN_TS}_insights.csv (Step 1) | {RUN_TS}_tasks.csv (here)
def _clean(r):
    """Make record JSON-serialisable (sets -> sorted lists)."""
    r = dict(r)
    if isinstance(r.get("industries"), set):
        r["industries"] = sorted(r["industries"])
    return r

order = [t[0] for t in T]
tasks_csv_path = OUT_DIR / f"{RUN_TS}_tasks.csv"

# two gold-answer columns -> gold_answer_json (valid JSON, for deterministic checks) and
# gold_answer_text (business-facing natural language). spec / gold_tool_path stay Python-repr.
def _csv_row(tid):
    r = _clean(records[tid])
    r["gold_answer_json"] = json.dumps(r.pop("gold_answer"))
    r["gold_answer_text"] = r.pop("gold_answer_nl", None)
    return r

tasks_df = pd.DataFrame([_csv_row(tid) for tid in order])
col_order = ["task_id", "type", "steps", "industries", "spec", "question",
             "gold_answer_json", "gold_answer_text", "gold_tool_path"]
tasks_df = tasks_df[[c for c in col_order if c in tasks_df.columns]
                    + [c for c in tasks_df.columns if c not in col_order]]
tasks_df.to_csv(tasks_csv_path, index=False)

print(f"wrote {tasks_csv_path}")
print(f"(corpus: {OUT_DIR / (RUN_TS + '_corpus.csv')} | insights: {OUT_DIR / (RUN_TS + '_insights.csv')})\n")
cost_report()

# End of notebook